# Modelo genérico · Rede Neural (MLP) para classificação binária

**Template reutilizável**: constrói, treina e avalia uma rede densa em Keras e depois **reimplementa a propagação direta em NumPy** (camada por camada e na forma vetorizada) com os pesos treinados, provando que as duas dão o mesmo resultado.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 5. Redes neurais e Deep Learning › cartão *Rede Neural (MLP)* (`/modelar/redes-neurais-e-deep-learning/#m-mlp`) |
| **Origem** | `C2_W1_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; reconhecimento dos dígitos 0 e 1 em imagens 20×20), revisado, corrigido e generalizado |
| **Tipo de problema** | Supervisionado · classificação binária |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `keras` 3 (com TensorFlow, JAX ou PyTorch como backend) |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, roda com dados tabulares sintéticos.
3. Se cada linha for uma **imagem achatada** (como os 400 pixels do original), informe `FORMATO_IMAGEM = (altura, largura)` para ver as grades de imagens com rótulo e previsão.

### O que mudou em relação ao notebook original
- O original treinava e "avaliava" no **mesmo conjunto**. Entraram divisão treino/validação/teste estratificada, *early stopping* na validação e as métricas do cartão no teste (ROC-AUC, PR-AUC, F1, log loss, matriz de confusão), além das curvas de perda por época.
- Camadas ocultas com **ReLU** por padrão (o cartão usa ReLU; o original usava sigmoide, que satura e treina mais devagar). A sigmoide continua disponível em `ATIVACAO_OCULTA`.
- Arquitetura parametrizável (`CAMADAS_OCULTAS`); a contagem de parâmetros por camada é calculada e conferida para qualquer arquitetura, não só 400-25-15-1.
- A propagação em NumPy virou genérica (`my_sequential` percorre a lista de camadas) e é comparada numericamente com o Keras; a `sigmoid` vinha de `autils.py` (ausente) e agora é definida no notebook.
- Keras 3 (`import keras`), como no cartão; `autils.py`, `public_tests.py` e as imagens do curso não são mais necessários.
- Sem a forma fixa 20×20, os índices fixos (`X[0]`, `X[500]`) e os rótulos do dataset original.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/transacoes.csv"; None = dados sintéticos
SEPARADOR = None
TEM_CABECALHO = True        # False se a primeira linha já é dado
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNA_ALVO = "alvo"        # coluna com duas classes
CLASSE_POSITIVA = None      # valor tratado como 1; None = usa 1 se o alvo já for 0/1
COLUNAS_FEATURES = None     # None = todas as demais colunas
FORMATO_IMAGEM = None       # ex.: (20, 20) se cada linha for uma imagem achatada; None = dados tabulares
TRANSPOR_IMAGEM = False     # True se os pixels estão em ordem de coluna (MATLAB/Octave), como no dataset original

# --- divisão e reprodutibilidade ---
TAMANHO_TESTE = 0.2
TAMANHO_VALIDACAO = 0.2     # fração do restante usada para validação (early stopping)
SEMENTE = 42

# --- rede ---
CAMADAS_OCULTAS = [25, 15]  # unidades por camada oculta (o original usava 25 e 15)
ATIVACAO_OCULTA = "relu"    # "relu" (padrão do cartão) ou "sigmoid" (como no original)
TAXA_APRENDIZADO = 0.001
EPOCAS = 200
TAMANHO_LOTE = 32
PACIENCIA = 15              # épocas sem melhora na validação antes de parar; None = sem early stopping
LIMIAR = 0.5

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, f1_score,
                             log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

keras.utils.set_random_seed(SEMENTE)      # semente do Python, do NumPy e do backend
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 2. Carregamento dos dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=1500, semente=SEMENTE):
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=n, n_features=12, n_informative=6, n_redundant=2,
                               class_sep=1.0, random_state=semente)
    df = pd.DataFrame(X, columns=[f"feature_{i + 1}" for i in range(X.shape[1])])
    df["alvo"] = y
    return df


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
if COLUNA_ALVO not in df.columns:
    raise KeyError(f"COLUNA_ALVO={COLUNA_ALVO!r} não existe. Colunas: {list(df.columns)}")
df = df.dropna(subset=[COLUNA_ALVO])
classes = sorted(df[COLUNA_ALVO].unique().tolist())
if len(classes) != 2:
    raise ValueError(f"Este template é binário; o alvo tem {len(classes)} classes. Use o template multiclasse.")
positiva = CLASSE_POSITIVA if CLASSE_POSITIVA is not None else (1 if set(classes) == {0, 1} else None)
if positiva is None:
    raise ValueError(f"Defina CLASSE_POSITIVA entre {classes}.")
outra = next(c for c in classes if c != positiva)

features = COLUNAS_FEATURES or [c for c in df.columns if c != COLUNA_ALVO]
X_df = df[features]
y = (df[COLUNA_ALVO] == positiva).astype("float32").to_numpy()
print("formato:", df.shape, "| proporção da classe positiva:", round(float(y.mean()), 3))
if FORMATO_IMAGEM is not None and np.prod(FORMATO_IMAGEM) != X_df.shape[1]:
    raise ValueError(f"FORMATO_IMAGEM={FORMATO_IMAGEM} não corresponde a {X_df.shape[1]} features.")

In [ ]:
def como_imagem(linha):
    img = np.asarray(linha, dtype=float).reshape(FORMATO_IMAGEM)
    return img.T if TRANSPOR_IMAGEM else img


def grade_imagens(X, titulos, n=64, semente=SEMENTE, titulo=""):
    """Mostra n linhas como imagens (só quando FORMATO_IMAGEM está definido)."""
    if FORMATO_IMAGEM is None:
        return
    rng = np.random.default_rng(semente)
    idx = rng.choice(len(X), size=min(n, len(X)), replace=False)
    lado = int(np.ceil(np.sqrt(len(idx))))
    fig, eixos = plt.subplots(lado, lado, figsize=(8, 8))
    for ax in eixos.flat:
        ax.set_axis_off()
    for ax, i in zip(eixos.flat, idx):
        ax.imshow(como_imagem(X[i]), cmap="gray")
        ax.set_title(titulos[i], fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


grade_imagens(X_df.to_numpy(), [str(v) for v in df[COLUNA_ALVO]], titulo="rótulo")
display(df.head())

## 3. Preparação
Divisão estratificada em treino, validação e teste; o pré-processamento (imputação, one-hot e **padronização, obrigatória para redes** segundo o cartão) é ajustado só no treino.

In [ ]:
def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


X_rest, X_te_df, y_rest, y_test = train_test_split(X_df, y, test_size=TAMANHO_TESTE, random_state=SEMENTE, stratify=y)
X_tr_df, X_va_df, y_train, y_val = train_test_split(X_rest, y_rest, test_size=TAMANHO_VALIDACAO,
                                                    random_state=SEMENTE, stratify=y_rest)
prep = montar_preprocessador(X_tr_df)
X_train = prep.fit_transform(X_tr_df).astype("float32")
X_val = prep.transform(X_va_df).astype("float32")
X_test = prep.transform(X_te_df).astype("float32")
n_features = X_train.shape[1]
print(f"treino {X_train.shape} | validação {X_val.shape} | teste {X_test.shape}")

## 4. Representação do modelo
Cada camada densa calcula $\mathbf{a}^{[l]} = g(\mathbf{a}^{[l-1]}\mathbf{W}^{[l]} + \mathbf{b}^{[l]})$. Com $s_{in}$ unidades na entrada e $s_{out}$ na saída, $\mathbf{W}$ tem forma $(s_{in}, s_{out})$ e $\mathbf{b}$ tem $s_{out}$ elementos (vetor 1-D, convenção do Keras). A saída tem **uma unidade sigmoide**: a probabilidade da classe positiva.

In [ ]:
model = keras.Sequential(
    [keras.Input(shape=(n_features,))]
    + [layers.Dense(u, activation=ATIVACAO_OCULTA, name=f"camada{i + 1}") for i, u in enumerate(CAMADAS_OCULTAS)]
    + [layers.Dense(1, activation="sigmoid", name="saida")],
    name="mlp_binaria",
)
model.summary()

# Parâmetros por camada = s_in * s_out (pesos) + s_out (vieses); confere com o Keras.
tamanhos = [n_features] + CAMADAS_OCULTAS + [1]
esperado = [s_in * s_out + s_out for s_in, s_out in zip(tamanhos[:-1], tamanhos[1:])]
assert esperado == [l.count_params() for l in model.layers], "contagem de parâmetros inesperada"
for camada in model.layers:
    W_, b_ = camada.get_weights()
    print(f"{camada.name}: W {W_.shape}, b {b_.shape}")

## 5. Treinamento

In [ ]:
model.compile(loss=keras.losses.BinaryCrossentropy(),
              optimizer=keras.optimizers.Adam(learning_rate=TAXA_APRENDIZADO),
              metrics=[keras.metrics.AUC(name="auc")])
callbacks = ([keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)]
             if PACIENCIA else [])
historico = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCAS,
                      batch_size=TAMANHO_LOTE, callbacks=callbacks, verbose=0)
print("épocas treinadas:", len(historico.history["loss"]))

plt.figure(figsize=(6, 3.5))
plt.plot(historico.history["loss"], label="treino")
plt.plot(historico.history["val_loss"], label="validação")
plt.xlabel("época"); plt.ylabel("log loss"); plt.legend(); plt.title("Curvas de perda")
plt.show()

## 6. Previsão e avaliação
A saída é uma probabilidade; como na regressão logística, ela é comparada a um limiar para virar classe.

In [ ]:
p_test = model.predict(X_test, verbose=0).ravel()
yhat_test = (p_test >= LIMIAR).astype(int)
metricas = {"acurácia": accuracy_score(y_test, yhat_test), "precisão": precision_score(y_test, yhat_test, zero_division=0),
            "recall": recall_score(y_test, yhat_test), "F1": f1_score(y_test, yhat_test),
            "ROC-AUC": roc_auc_score(y_test, p_test), "PR-AUC": average_precision_score(y_test, p_test),
            "log loss": log_loss(y_test, p_test)}
display(pd.Series(metricas, name="teste").round(4).to_frame().T)
ConfusionMatrixDisplay.from_predictions(y_test, yhat_test, colorbar=False)
plt.title(f"Matriz de confusão (teste, limiar {LIMIAR})")
plt.show()

## 7. Propagação direta em NumPy
Uma camada densa, unidade a unidade: para cada unidade $j$, $z_j = \mathbf{w}_j\cdot\mathbf{a}_{in} + b_j$ e $a_j = g(z_j)$, onde $\mathbf{w}_j$ é a **coluna** $j$ de $\mathbf{W}$. Na forma vetorizada, todas as unidades e todos os exemplos de uma vez: $\mathbf{Z} = \mathbf{A}_{in}\mathbf{W} + \mathbf{b}$ (o NumPy replica $\mathbf{b}$ nas $m$ linhas por *broadcasting*).

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def relu(z):
    return np.maximum(0.0, z)


ATIVACOES = {"sigmoid": sigmoid, "relu": relu, "linear": lambda z: z}


def my_dense(a_in, W, b, g):
    """Camada densa para UM exemplo. a_in: (n,), W: (n, j), b: (j,) → a_out: (j,)."""
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:, j]                     # pesos da unidade j = coluna j de W
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return a_out


def my_dense_v(A_in, W, b, g):
    """Camada densa vetorizada. A_in: (m, n), W: (n, j), b: (j,) ou (1, j) → (m, j)."""
    return g(np.matmul(A_in, W) + b)


def my_sequential(x, camadas, densa=my_dense_v):
    """Encadeia as camadas: camadas = [(W, b, g), ...]."""
    a = x
    for W, b, g in camadas:
        a = densa(a, W, b, g)
    return a


# Os mesmos casos de teste do original, com as saídas esperadas que ele documentava.
x_tst = 0.1 * np.arange(1, 3, 1).reshape(2,)
W_tst = 0.1 * np.arange(1, 7, 1).reshape(2, 3)
b_tst = 0.1 * np.arange(1, 4, 1).reshape(3,)
assert np.allclose(my_dense(x_tst, W_tst, b_tst, sigmoid), [0.54735762, 0.57932425, 0.61063923])
X_tst = 0.1 * np.arange(1, 9, 1).reshape(4, 2)
assert np.allclose(my_dense_v(X_tst, W_tst, b_tst.reshape(1, 3), sigmoid)[-1], [0.62010643, 0.67699586, 0.72908792])
print("my_dense e my_dense_v conferem com as saídas esperadas do original")

In [ ]:
# Copia os pesos treinados do Keras e compara as duas implementações com a rede do Keras.
camadas_np = [(*c.get_weights(), ATIVACOES[c.get_config()["activation"]]) for c in model.layers]

p_numpy_v = my_sequential(X_test, camadas_np).ravel()                               # vetorizada
p_numpy_loop = np.array([my_sequential(x, camadas_np, densa=my_dense)[0] for x in X_test[:50]])  # unidade a unidade
print("máx. |Keras − NumPy vetorizado|:", float(np.max(np.abs(p_test - p_numpy_v))))
print("máx. |Keras − NumPy em laço|  :", float(np.max(np.abs(p_test[:50] - p_numpy_loop))))
assert np.allclose(p_test, p_numpy_v, atol=1e-4) and np.allclose(p_test[:50], p_numpy_loop, atol=1e-4)
print("concordância das classes previstas:", np.mean((p_numpy_v >= LIMIAR) == yhat_test))

In [ ]:
# Grades de imagens (só com FORMATO_IMAGEM): rótulo, previsão do Keras e previsão do NumPy; e um erro.
if FORMATO_IMAGEM is not None:
    X_img = X_te_df.to_numpy()
    rot = [f"{int(a)},{int(b)},{int(c)}" for a, b, c in zip(y_test, yhat_test, p_numpy_v >= LIMIAR)]
    grade_imagens(X_img, rot, titulo="rótulo, previsão Keras, previsão NumPy")
    erros = np.flatnonzero(yhat_test != y_test)
    if len(erros):
        i = erros[0]
        plt.figure(figsize=(1.5, 1.5))
        plt.imshow(como_imagem(X_img[i]), cmap="gray"); plt.axis("off")
        plt.title(f"{int(y_test[i])}, {yhat_test[i]}")
        plt.show()
else:
    erros = np.flatnonzero(yhat_test != y_test)
    print(f"{len(erros)} erros em {len(y_test)} exemplos de teste; alguns deles:")
    display(X_te_df.iloc[erros[:5]].assign(real=y_test[erros[:5]], probabilidade=p_test[erros[:5]].round(3)))

### 7.1 Broadcasting em uma linha
Operações elemento a elemento comparam as formas da direita para a esquerda: duas dimensões são compatíveis se forem iguais ou se uma delas for 1, que é então "esticada". É isso que permite somar o vetor de vieses `b` (forma `(j,)`) a `A_in @ W` (forma `(m, j)`).

In [ ]:
a = np.array([1, 2, 3, 4]).reshape(-1, 1)   # (4, 1)
b = np.array([1, 2, 3]).reshape(1, -1)      # (1, 3)
print((a + b).shape)                        # (4, 3)
print(a + b)

## 8. Previsão para novos dados

In [ ]:
novos = X_df.sample(3, random_state=SEMENTE)     # >>> ADAPTAR: DataFrame com as mesmas colunas de `features`
proba = model.predict(prep.transform(novos).astype("float32"), verbose=0).ravel()
display(novos.iloc[:, :8].assign(probabilidade=proba.round(4), previsao=[positiva if p >= LIMIAR else outra for p in proba]))
model.save("mlp_binaria.keras")             # formato nativo do Keras 3 (o .h5 é legado)

## 9. Próximos passos (segundo o cartão do sistema)
- Hiperparâmetros que mais importam: arquitetura, taxa de aprendizado, tamanho do lote, épocas, *dropout*, *weight decay* e *early stopping*.
- Acompanhe sempre as curvas de treino × validação por época.
- Em dados **tabulares** de porte médio, **prefira boosting**, a menos que a rede mostre ganho consistente em validação cruzada.